# 📊 Event Study：大跌后抄底是否有效？

本研究旨在回答：

> 当市场出现大幅下跌时，抄底是否真的能提高收益？还是只是“接飞刀”？

---

## 🧠 核心研究框架

本策略从 6 个核心维度定义：

---

## 1️⃣ 大跌定义（Event）

用历史分布定义“极端下跌日”：

- Worst **10%**（较大下跌）
- Worst **5%**（极端下跌）
- Worst **1%**（恐慌级下跌）

📌 定义方式：
- 使用 daily return（收盘→收盘）
- 按历史分位数划分

---

## 2️⃣ 指数选择（Market）

用于对比不同市场结构：

- 🇨🇳 上证综指（SSE: 000001.SS） → 散户主导 / 情绪更强
- 🇺🇸 标普500（SPY） → 机构主导 / 更成熟市场

📌 目的：
- 比较不同市场中“抄底行为”的有效性差异

---

## 3️⃣ 入场方式（Entry）

统一为：

> **事件发生后的次日开盘买入**

📌 原因：
- 更贴近真实投资者行为
- 避免未来函数（look-ahead bias）

---

## 4️⃣ 持有期（Holding Period）

模拟不同投资风格：

- 1 天 → 超短线（情绪交易）
- 3 / 5 天 → 短期反弹
- 10 天 → 波段交易
- 20 天 → 一个月
- 60 天 → 中期持有（~3个月）

📌 目的：
- 判断“抄底收益”是否依赖持有时间

---

## 5️⃣ 市场状态（Regime）

根据趋势划分：

- MA200 上方 → 上升趋势（牛市环境）
- MA200 下方 → 下降趋势（熊市环境）

📌 目的：
- 判断：  
  👉 大跌是“黄金坑”，还是“下跌中继”

---

## 6️⃣ 波动率环境（VIX，可选）

（仅适用于美股）

- 高 VIX → 恐慌市场
- 低 VIX → 平稳市场

📌 目的：
- 判断在“恐慌极端时刻”抄底是否更有效

---

## 📈 输出指标（Metrics）

每种组合下计算：

- 平均收益（Average Return）
- 中位数收益（Median Return）
- 胜率（Win Rate）
- 最大回撤（Max Drawdown）

并与随机买入（Random Baseline）对比：

> 👉 判断是否存在“统计优势”

---

## 🎯 核心研究问题

1. 大跌后买入，收益是否高于随机？
2. 不同持有期结果是否完全不同？
3. 牛市 vs 熊市，大跌的意义是否不同？
4. A股 vs 美股，抄底是否有本质差异？
5. 极端恐慌（VIX高）是否是更好的买点？

---

## 🧠 核心假设（Hypothesis）

- H1：大跌后存在短期反弹（mean reversion）
- H2：但在熊市中，大跌更可能是趋势延续
- H3：美股比A股更容易形成“有效抄底”
- H4：持有期越长，结果越稳定，但边际收益下降

---

## 🔥 研究目标（用于视频表达）

本研究最终想回答：

> **“黄金坑”是真的存在，还是大多数人只是买在“坑里”？**

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

# ========= CONFIG =========
TICKERS = {
    "SPY": "SPY",
    "SSE": "000001.SS",   # 上证综指
}

START = "2004-01-01"
HOLDINGS = [1, 3, 5, 10, 20, 60]
PCTS = [1, 5, 10]

np.random.seed(42)  # 固定随机结果，方便每次跑出来的 random baseline 一致


# ========= DATA =========
def load_data(ticker):
    df = yf.download(
        ticker,
        start=START,
        progress=False,      # 不显示下载进度条
        auto_adjust=False    # 不自动复权；保留原始 OHLC，更适合当前短中期事件研究
    )

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)  # 如果列名是两层结构，只取第一层

    cols = ["Open", "High", "Low", "Close"]
    df = df[cols].dropna().copy()

    df["ret"] = df["Close"].pct_change()             # 用收盘到收盘收益定义“大跌日”
    df["ma200"] = df["Close"].rolling(200).mean()    # 200日均线
    df["regime"] = np.where(df["Close"] > df["ma200"], "above", "below")
    df.loc[df["ma200"].isna(), "regime"] = np.nan

    return df


# ========= EVENT =========
def get_event_dates(df, pct):
    threshold = np.percentile(df["ret"].dropna(), pct)
    mask = (df["ret"] <= threshold) & df["ma200"].notna()
    return df.index[mask]


# ========= SINGLE TRADE =========
def calc_trade(df, event_date, holding):
    i = df.index.get_loc(event_date)

    entry_i = i + 1
    exit_i = entry_i + holding - 1
    # holding=1: 次日开盘买，次日收盘卖
    # holding=5: 次日开盘买，第5个交易日收盘卖

    if exit_i >= len(df):
        return None

    entry_price = df.iloc[entry_i]["Open"]
    exit_price = df.iloc[exit_i]["Close"]

    if pd.isna(entry_price) or pd.isna(exit_price) or entry_price <= 0:
        return None

    ret = exit_price / entry_price - 1

    period_low = df.iloc[entry_i:exit_i + 1]["Low"].min()
    max_dd = period_low / entry_price - 1

    return {
        "event_date": event_date,
        "entry_date": df.index[entry_i],
        "exit_date": df.index[exit_i],
        "regime": df.loc[event_date, "regime"],
        "return": ret,
        "max_dd": max_dd,
        "win": int(ret > 0),
    }


# ========= EVENT BACKTEST =========
def run_event_backtest(df, pct, holding):
    events = get_event_dates(df, pct)
    rows = []

    for dt in events:
        trade = calc_trade(df, dt, holding)
        if trade is not None:
            rows.append(trade)

    return pd.DataFrame(rows)


# ========= RANDOM BASELINE =========
def run_random_baseline(df, n_trades, holding, regime=None):
    valid_idx = np.arange(1, len(df) - holding + 1)

    if regime is not None:
        valid_idx = [i for i in valid_idx if df.iloc[i - 1]["regime"] == regime]

    if len(valid_idx) == 0 or n_trades == 0:
        return pd.DataFrame()

    sampled_idx = np.random.choice(valid_idx, size=n_trades, replace=True)

    rows = []
    for entry_i in sampled_idx:
        exit_i = entry_i + holding - 1

        entry_price = df.iloc[entry_i]["Open"]
        exit_price = df.iloc[exit_i]["Close"]
        period_low = df.iloc[entry_i:exit_i + 1]["Low"].min()

        ret = exit_price / entry_price - 1
        max_dd = period_low / entry_price - 1

        rows.append({
            "return": ret,
            "max_dd": max_dd,
            "win": int(ret > 0),
        })

    return pd.DataFrame(rows)


# ========= SUMMARY =========
def summarize(trades):
    if trades.empty:
        return {
            "count": 0,
            "avg_return": np.nan,
            "median_return": np.nan,
            "win_rate": np.nan,
            "avg_max_dd": np.nan,
        }

    return {
        "count": len(trades),
        "avg_return": trades["return"].mean(),
        "median_return": trades["return"].median(),
        "win_rate": trades["win"].mean(),
        "avg_max_dd": trades["max_dd"].mean(),
    }


def print_summary(title, stats_event, stats_random):
    print(f"\n{title}")
    print("-" * len(title))

    print(
        f"Event  | n={stats_event['count']:<4} "
        f"avg={stats_event['avg_return']:.2%} "
        f"median={stats_event['median_return']:.2%} "
        f"win={stats_event['win_rate']:.2%} "
        f"avgMDD={stats_event['avg_max_dd']:.2%}"
    )

    print(
        f"Random | n={stats_random['count']:<4} "
        f"avg={stats_random['avg_return']:.2%} "
        f"median={stats_random['median_return']:.2%} "
        f"win={stats_random['win_rate']:.2%} "
        f"avgMDD={stats_random['avg_max_dd']:.2%}"
    )


# ========= MAIN =========
results = []

for name, ticker in TICKERS.items():
    print(f"\n{'='*60}")
    print(f"{name} ({ticker})")
    print(f"{'='*60}")

    df = load_data(ticker)

    for pct in PCTS:
        for holding in HOLDINGS:
            # -------- ALL --------
            event_trades = run_event_backtest(df, pct, holding)
            random_trades = run_random_baseline(df, len(event_trades), holding)

            event_stats = summarize(event_trades)
            random_stats = summarize(random_trades)

            print_summary(
                title=f"Worst {pct}% days | Holding {holding}d | ALL",
                stats_event=event_stats,
                stats_random=random_stats
            )

            results.append({
                "market": name,
                "pct": pct,
                "holding": holding,
                "regime": "ALL",
                "event_n": event_stats["count"],
                "event_avg_return": event_stats["avg_return"],
                "event_median_return": event_stats["median_return"],
                "event_win_rate": event_stats["win_rate"],
                "event_avg_mdd": event_stats["avg_max_dd"],
                "random_n": random_stats["count"],
                "random_avg_return": random_stats["avg_return"],
                "random_median_return": random_stats["median_return"],
                "random_win_rate": random_stats["win_rate"],
                "random_avg_mdd": random_stats["avg_max_dd"],
            })

            # -------- MA200 ABOVE --------
            event_above = event_trades[event_trades["regime"] == "above"]
            random_above = run_random_baseline(df, len(event_above), holding, regime="above")

            event_stats = summarize(event_above)
            random_stats = summarize(random_above)

            print_summary(
                title=f"Worst {pct}% days | Holding {holding}d | MA200 ABOVE",
                stats_event=event_stats,
                stats_random=random_stats
            )

            results.append({
                "market": name,
                "pct": pct,
                "holding": holding,
                "regime": "ABOVE",
                "event_n": event_stats["count"],
                "event_avg_return": event_stats["avg_return"],
                "event_median_return": event_stats["median_return"],
                "event_win_rate": event_stats["win_rate"],
                "event_avg_mdd": event_stats["avg_max_dd"],
                "random_n": random_stats["count"],
                "random_avg_return": random_stats["avg_return"],
                "random_median_return": random_stats["median_return"],
                "random_win_rate": random_stats["win_rate"],
                "random_avg_mdd": random_stats["avg_max_dd"],
            })

            # -------- MA200 BELOW --------
            event_below = event_trades[event_trades["regime"] == "below"]
            random_below = run_random_baseline(df, len(event_below), holding, regime="below")

            event_stats = summarize(event_below)
            random_stats = summarize(random_below)

            print_summary(
                title=f"Worst {pct}% days | Holding {holding}d | MA200 BELOW",
                stats_event=event_stats,
                stats_random=random_stats
            )

            results.append({
                "market": name,
                "pct": pct,
                "holding": holding,
                "regime": "BELOW",
                "event_n": event_stats["count"],
                "event_avg_return": event_stats["avg_return"],
                "event_median_return": event_stats["median_return"],
                "event_win_rate": event_stats["win_rate"],
                "event_avg_mdd": event_stats["avg_max_dd"],
                "random_n": random_stats["count"],
                "random_avg_return": random_stats["avg_return"],
                "random_median_return": random_stats["median_return"],
                "random_win_rate": random_stats["win_rate"],
                "random_avg_mdd": random_stats["avg_max_dd"],
            })

# ========= FINAL TABLE =========
df_results = pd.DataFrame(results)

# 增加超额指标，方便直接比较
df_results["excess_avg_return"] = df_results["event_avg_return"] - df_results["random_avg_return"]
df_results["excess_win_rate"] = df_results["event_win_rate"] - df_results["random_win_rate"]
df_results["excess_avg_mdd"] = df_results["event_avg_mdd"] - df_results["random_avg_mdd"]

print("\n" + "=" * 80)
print("FINAL SUMMARY TABLE")
print("=" * 80)
print(df_results)

# 保存 csv
df_results.to_csv("event_study_summary.csv", index=False)

print("\nSaved to: event_study_summary.csv")




SPY (SPY)

Worst 1% days | Holding 1d | ALL
--------------------------------
Event  | n=56   avg=0.53% median=0.33% win=58.93% avgMDD=-2.45%
Random | n=56   avg=-0.02% median=0.05% win=58.93% avgMDD=-0.60%

Worst 1% days | Holding 1d | MA200 ABOVE
----------------------------------------
Event  | n=5    avg=0.43% median=0.27% win=60.00% avgMDD=-1.73%
Random | n=5    avg=-0.21% median=-0.17% win=20.00% avgMDD=-0.59%

Worst 1% days | Holding 1d | MA200 BELOW
----------------------------------------
Event  | n=51   avg=0.53% median=0.53% win=58.82% avgMDD=-2.53%
Random | n=51   avg=0.56% median=0.42% win=62.75% avgMDD=-1.00%

Worst 1% days | Holding 3d | ALL
--------------------------------
Event  | n=56   avg=0.50% median=0.98% win=53.57% avgMDD=-4.27%
Random | n=56   avg=0.37% median=0.63% win=67.86% avgMDD=-0.88%

Worst 1% days | Holding 3d | MA200 ABOVE
----------------------------------------
Event  | n=5    avg=0.11% median=-0.89% win=40.00% avgMDD=-2.18%
Random | n=5    avg=0.53% 

In [8]:
print(df_results[df_results["regime"] == "ALL"].sort_values("excess_avg_return", ascending=False))

    market  pct  holding regime  event_n  event_avg_return  \
15     SPY    1       60    ALL       56          0.053477   
33     SPY    5       60    ALL      277          0.037412   
51     SPY   10       60    ALL      538          0.034020   
54     SSE    1        1    ALL       54          0.014045   
66     SSE    1       20    ALL       54         -0.001090   
57     SSE    1        3    ALL       54          0.009557   
63     SSE    1       10    ALL       54          0.013383   
45     SPY   10       10    ALL      542          0.008160   
48     SPY   10       20    ALL      541          0.013079   
12     SPY    1       20    ALL       56          0.013912   
96     SSE   10        5    ALL      511          0.004664   
0      SPY    1        1    ALL       56          0.005252   
24     SPY    5        5    ALL      278          0.004209   
72     SSE    5        1    ALL      263          0.004906   
81     SSE    5       10    ALL      262          0.002903   
90     S

In [11]:
for name, ticker in TICKERS.items():
    print(f"\n{name} thresholds:")

    df = load_data(ticker)

    for pct in PCTS:
        threshold = np.percentile(df["ret"].dropna(), pct)
        print(f"  worst {pct}%: {threshold:.2%}")


SPY thresholds:
  worst 1%: -3.42%
  worst 5%: -1.73%
  worst 10%: -1.15%

SSE thresholds:
  worst 1%: -4.79%
  worst 5%: -2.26%
  worst 10%: -1.51%
